In [ ]:
# Any parametrized values for times square?
import rubin_nights.dayobs_utils as rn_dayobs

day_obs = rn_dayobs.today_day_obs()
day_obs = rn_dayobs.day_obs_str_to_int(day_obs)

data_dir = "/Users/lynnej/opsim/fbs_5.3"

# fO (diagnostic notebook)

This demonstrates how to calculate fO, as a component of the fE tier-1 metric (ROSP-6). This is not an official metric page. 

fO is a measurement for the Area and Nv values at the end of survey. 

It can be split into two parts: 
* fO_area - the amount of area which is covered by a minimum number of 750 visits
* fO_Nv - the median number of visits per point over the best-covered 18,000 sq deg 


Because these numbers necessarily include a simulation of the remainder of the survey (the fO value is related to an estimate of total survey throughput), reference points include:
* a chosen simulation for the entire survey (reference sim)
* a 'bespoke' simulation that starts from a specified on-sky visit acquisition point, then continues with current expected performance
   * the bespoke simulation could be the same as the chosen simulation 
* on-sky visits up to today + the bespoke simulation for the remainder of the survey ('chimera')

Other comparisons will use a 'bespoke' simulation that starts at today and continues to the end of survey.

Jira task: ROSP-21. Epic ROSP-06.<br>
Performance Tracking Hub: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696<br>
Steward: Lynne Jones. <br>
Method: in development. Reference simulations not yet determined.<br>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import colorcet
import healpy as hp
import sqlite3
import rubin_sim.maf as maf
from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
from astropy.time import Time

# Remove warnings about conversion utc/tai in the future
import warnings
from erfa import ErfaWarning

warnings.filterwarnings("ignore", category=ErfaWarning)

In [ ]:
# Get parameters from times square
day_obs = int(day_obs)

In [ ]:
endpoints = connections.get_clients()

In [ ]:
start = 20260617
visits = endpoints["consdb"].query(
    "select visit_id, day_obs, obs_start_mjd, s_ra, s_dec, sky_rotation, band, exp_time, scheduler_note from cdb_lsstcam.visit1 "
    f"where day_obs >= {start} and img_type = 'science' and can_see_sky=True"
)
visits.rename(
    {
        "visit_id": "observationId",
        "obs_start_mjd": "observationStartMJD",
        "s_ra": "fieldRA",
        "s_dec": "fieldDec",
        "sky_rotation": "rotSkyPos",
        "exp_time": "visitExposureTime",
    },
    axis=1,
    inplace=True,
)

In [ ]:
# Pinned run for comparison
sim_db = "baseline_v5.3.4_10yrs"

# For running on any USDF-based RSP or in times square, use the copy at USDF:
if "usdf" in os.getenv("EXTERNAL_INSTANCE_URL", ""):
    data_dir = "/sdf/data/rubin/shared/fbs_sims/sims_featureScheduler_runs5.3/baseline"

conn = sqlite3.connect(os.path.join(data_dir, sim_db + ".db"))
sim = pd.read_sql(
    "select observationId, observationStartMJD, fieldRA, fieldDec, rotSkyPos, band, visitExposureTime, scheduler_note from observations",
    conn,
)
sim["day_obs"] = rn_dayobs.mjd_to_day_obs_array(sim["observationStartMJD"])

In [ ]:
# More recent bespoke run? Might be the same depending on when pinned run created.
switchover = 20260617
bespoke_sim = "baseline_v5.3.4_10yrs"

conn = sqlite3.connect(os.path.join(data_dir, bespoke_sim + ".db"))
bespoke = pd.read_sql(
    "select observationId, observationStartMJD, fieldRA, fieldDec, rotSkyPos, band, visitExposureTime, scheduler_note from observations",
    conn,
)
bespoke["day_obs"] = rn_dayobs.mjd_to_day_obs_array(bespoke["observationStartMJD"])

In [ ]:
chimera = pd.concat(
    [visits.query("day_obs < @day_obs"), bespoke.query("day_obs >= @day_obs")]
)

In [ ]:
print(f"Using {sim_db} as the (full survey) comparison simulation.")
print(
    f"Using {bespoke_sim} as the bespoke simulation (real visits, continuation in simulation after {switchover})."
)
print(
    f"Using on-sky visits from {start} to {day_obs} combined with {bespoke_sim} as chimera simulation."
)

In [ ]:
# We need to count the number of visits for WFD vs. non-WFD
# in order to correct the calculated Nv values to the equivalent SRD value.
# The SRD value assumes that 90% of the total visits are for WFD; the survey strategy assigns fewer.
nside = 64
allsky_slicer = maf.HealpixSlicer(nside=nside)
wfd_slicer = maf.make_wfd_subset_slicer(nside=nside)
wfd_hpix = np.where(wfd_slicer.mask, 0, 1)

footprint_stacker = maf.WFDlabelStacker(
    footprint=wfd_hpix,
    fp_threshold=0.4,
    area_id_name="WFD",
    exclude_dd=True,
    ra_col="fieldRA",
    dec_col="fieldDec",
    note_col="scheduler_note",
)


sim = pd.DataFrame(footprint_stacker.run(sim.to_records(index=False)))
bespoke = pd.DataFrame(footprint_stacker.run(bespoke.to_records(index=False)))
chimera = pd.DataFrame(footprint_stacker.run(chimera.to_records(index=False)))

In [ ]:
fO_sky = maf.fOBatch(run_name=f"on-sky-{day_obs}", slicer=allsky_slicer)
fO_bespoke = maf.fOBatch(run_name=bespoke_sim, slicer=allsky_slicer)
fO_sim = maf.fOBatch(run_name=sim_db, slicer=allsky_slicer)

In [ ]:
g = maf.MetricBundleGroup(fO_sky, None, out_dir="temp")
g.run_current("", sim_data=chimera.to_records())

In [ ]:
g = maf.MetricBundleGroup(fO_bespoke, None, out_dir="temp")
g.run_current("", sim_data=bespoke.to_records())

In [ ]:
g = maf.MetricBundleGroup(fO_sim, None, out_dir="temp")
g.run_current("", sim_data=sim.to_records())

In [ ]:
k_sky = list(fO_sky.keys())[0]
wfd_ratio = round(len(chimera.query("area_id == 'WFD'")) / len(chimera), 3)
ll = pd.DataFrame(fO_sky[k_sky].summary_values["fONv"])
ll = pd.concat(
    [
        ll,
        pd.DataFrame(
            ["Area_750", round(fO_sky[k_sky].summary_values["fOArea_750"], 1)],
            index=["name", "value"],
        ).T,
        pd.DataFrame(["wfd_ratio", wfd_ratio], index=["name", "value"]).T,
    ]
).reset_index(drop=True)
ll["data"] = "chimera"
ll

In [ ]:
k_bespoke = list(fO_bespoke.keys())[0]
wfd_ratio = round(len(bespoke.query("area_id == 'WFD'")) / len(bespoke), 3)
ll2 = pd.DataFrame(fO_bespoke[k_bespoke].summary_values["fONv"])
ll2 = pd.concat(
    [
        ll2,
        pd.DataFrame(
            ["Area_750", round(fO_bespoke[k_bespoke].summary_values["fOArea_750"], 1)],
            index=["name", "value"],
        ).T,
        pd.DataFrame(["wfd_ratio", wfd_ratio], index=["name", "value"]).T,
    ]
).reset_index(drop=True)
ll2["data"] = "bespoke"

In [ ]:
k_sim = list(fO_sim.keys())[0]
wfd_ratio = round(len(sim.query("area_id == 'WFD'")) / len(sim), 3)
ll3 = pd.DataFrame(fO_sim[k_sim].summary_values["fONv"])
ll3 = pd.concat(
    [
        ll3,
        pd.DataFrame(
            ["Area_750", round(fO_sim[k_sim].summary_values["fOArea_750"], 1)],
            index=["name", "value"],
        ).T,
        pd.DataFrame(["wfd_ratio", wfd_ratio], index=["name", "value"]).T,
    ]
).reset_index(drop=True)
ll3["data"] = "sim"
ll = pd.concat([ll, ll2, ll3]).set_index(["data", "name"])

In [ ]:
x = ll.loc[("chimera", "MedianNvis")] / ll.loc[("sim", "MedianNvis")]
print("Nv Ratio of chimera to ref sim: ", round(x.value, 2))
x = ll.loc[("chimera", "Area_750")] / ll.loc[("sim", "Area_750")]
print("Area Ratio of chimera to ref sim: ", round(x.value, 2))

x = ll.loc[("chimera", "MedianNvis")] / (750 * ll.loc[("chimera", "wfd_ratio")] / 0.9)
print(
    "Nv Ratio of chimera to benchmark 750 visits w/ WFD-correction: ", round(x.value, 3)
)
x = ll.loc[("chimera", "Area_750")] / (15000 * ll.loc[("chimera", "wfd_ratio")] / 0.9)
print(
    "Area Ratio of chimera to benchmark 15,000 sq deg w/ WFD-correction: ",
    round(x.value, 3),
)

print("")
print(f"Baseline simulation is {sim_db}")
print(f"Bespoke simulation is {bespoke_sim}")
if bespoke_sim == sim_db:
    print("Currently using identical bespoke and reference simulations")
ll

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(8, 6))
colors = colorcet.glasbey_category10

for count, bundle in enumerate([fO_sim[k_sim], fO_bespoke[k_bespoke], fO_sky[k_sky]]):
    if (bespoke_sim == sim_db) and count == 1:
        continue
    name = bundle.run_name
    scale = hp.nside2pixarea(bundle.slicer.nside, degrees=True) / 1000.0

    nvis_sorted = np.sort(bundle.metric_values.filled(0))
    area = np.arange(29000, 0, -5)
    medianval = np.zeros(len(area))
    minval = np.zeros(len(area))
    for i, asky in enumerate(area):
        npix__asky = int(
            np.ceil(asky / hp.nside2pixarea(bundle.slicer.nside, degrees=True))
        )
        # Find the Asky's worth of healpixels with the largest # of visits.
        nvis__asky = nvis_sorted[-npix__asky:]
        medianval[i] = np.median(nvis__asky)
        minval[i] = np.min(nvis__asky)
    idx_15k = np.where(abs(area - 15000) < 0.001)[0]
    idx_18k = np.where(abs(area - 18000) < 0.001)[0]
    axes.plot(
        medianval,
        area / 1000,
        color=colors[count],
        linestyle="-",
        label=f"{name}: median Nv 15k sq deg {medianval[idx_15k].item():.0f}",
    )
    axes.plot(
        minval,
        area / 1000,
        linestyle=":",
        color=colors[count],
        label=f"{name}: minimum Nv 15k sq deg {minval[idx_15k].item():.0f}",
    )

axes.legend(loc=(1.01, 0.1))
axes.fill_between([750, 825], y1=[0], y2=[30], color="gray", alpha=0.2)
axes.fill_betweenx([15, 18], x1=[0], x2=[1200], color="gray", alpha=0.2)
axes.set_xlim(0, 1200)
axes.set_ylim(0, 29)
axes.set_xlabel("Number of visits", fontsize="x-large")
_ = axes.set_ylabel("Area (100s of square degrees) ", fontsize="x-large")

In [ ]:
x = rn_dayobs.day_obs_list(
    Time(visits.observationStartMJD.min(), format="mjd", scale="tai"),
    Time(sim.observationStartMJD.max(), format="mjd", scale="tai"),
)
# Cut down the length
x = x[0:365]
xi = np.arange(0, len(x), 1)

b = sim.groupby("day_obs").agg({"observationStartMJD": "count"})
b = b.reindex(x, fill_value=0)
plt.plot(
    xi, b.observationStartMJD.cumsum(), linestyle="-", linewidth=2, label=f"Sim visits"
)

a = visits.groupby("day_obs").agg({"observationStartMJD": "count"})
a = a.reindex(x, fill_value=0)
# Cut real-visits line back to today
a = a.query("index < @day_obs")
xii = np.arange(0, len(a), 1)
plt.plot(
    xii,
    a.observationStartMJD.cumsum(),
    linestyle="-",
    linewidth=2,
    label=f"On-sky visits",
)
plt.axvline(len(a), color="black", linestyle="-.")

c = chimera.groupby("day_obs").agg({"observationStartMJD": "count"})
c = c.reindex(x, fill_value=0)
plt.plot(
    xi, c.observationStartMJD.cumsum(), linestyle=":", linewidth=2, label=f"Chimera"
)

interval = 15
plt.xticks(ticks=xi[::interval], labels=x[::interval], rotation=90)
plt.legend()
plt.ylabel("Number of visits", fontsize="large")
plt.xlabel("DayObs", fontsize="large")

_ = plt.grid(alpha=0.3)
_ = plt.xlim(-0.5, None)

In [ ]:
# Potentially just check an extra simulation
run_extra_sim = False
if run_extra_sim:
    esim_db = "baseline_v5.0.1_10yrs"

    # Paste your own location of the simulation here if needed
    data_dir = "/Users/lynnej/opsim/fbs_5.0"
    # For running on USDF, use a shared location
    if "usdf" in os.getenv("EXTERNAL_INSTANCE_URL", ""):
        if os.getenv("RUBIN_SIM_DATA_DIR") is None:
            # Use shared RUBIN_SIM_DATA_DIR
            os.environ["RUBIN_SIM_DATA_DIR"] = (
                "/sdf/data/rubin/shared/fbs_sims/sims_featureScheduler_runs5.0/baseline"
            )

    conn = sqlite3.connect(os.path.join(data_dir, esim_db + ".db"))
    esim = pd.read_sql(
        "select observationId, observationStartMJD, fieldRA, fieldDec, rotSkyPos, band, visitExposureTime from observations",
        conn,
    )
    esim["day_obs"] = rn_dayobs.mjd_to_day_obs_array(esim["observationStartMJD"])

    fO = maf.fOBatch(run_name=esim_db)
    g = maf.MetricBundleGroup(fO, None, out_dir="temp")
    g.run_current("", sim_data=esim.to_records())

    k = list(fO.keys())[0]
    ll4 = pd.DataFrame(fO[k].summary_values["fONv"])
    ll4 = pd.concat(
        [
            ll4,
            pd.DataFrame(
                ["Area_750", round(fO[k].summary_values["fOArea_750"], 1)],
                index=["name", "value"],
            ).T,
        ]
    ).reset_index(drop=True)
    ll4["data"] = "esim"
    ll4 = ll4.set_index(["data", "name"])

    x = ll4.loc[("esim", "MedianNvis")] / (750 * 0.9)
    print(
        f"Nv Ratio of {esim_db} to benchmark 750 visits (fractional fO with 90% correction): ",
        round(x.value, 2),
    )
    x = ll4.loc[("esim", "Area_750")] / (15000 * 0.9)
    print(
        f"Area Ratio of {esim_db} to benchmark 15,000 sq deg (fractional fO with 90% correction): ",
        round(x.value, 3),
    )